DSA506 Visual Analytics and Communications  
Michelle Richardson  
Test 1  

**Infrastructure EDA**  

In [55]:

import pandas as pd
import folium
from folium.plugins import HeatMap
import plotly.express as px
from IPython.display import display, IFrame

print(f"Folium version: {folium.__version__}")
print("All libraries loaded.")

Folium version: 0.20.0
All libraries loaded.


Mapping tasks:  
*Display all infrastructure points using color-coded markers based on operational status (for example, green for fully operational, orange for partially operational, red for non-operational).  
*Add a meaningful layer or clustering strategy that communicates the geographic concentration of damage, such as municipality grouping, facility type layers, or damage severity heatmaps.  
*Include popup or tooltip information for each facility showing at minimum: facility name, type, operational status, damage severity, and population served.  
*Identify geographic gaps in operational infrastructure. In particular, assess which areas lack an operational hospital or water treatment facility within a serviceable radius. You must define and justify the distance threshold you choose (for example, 30 km, 50 km, or another value), explaining why it is appropriate in a post-hurricane context with degraded road networks.  
*Provide a written interpretation (one to two paragraphs) of what the map reveals about the spatial distribution of damage, which municipalities appear most underserved, and what the map suggests about where relief resources should be prioritized.


In [56]:
df = pd.read_csv("isla_coralina_infrastructure.csv")
df.head()
#df

,facility_id,facility_name,facility_type,municipality,latitude,longitude,operational_status,damage_severity,population_served,date_last_update,has_generator,road_access
0,IC-0001,Central Supply Distribution Center Pue-001,Supply Distribution Center,Puerto Nuevo,20.029503,-75.843798,Fully Operational,1,3455,2025-09-16,No,Yes
1,IC-0002,Mountain Bridge Pue-002,Bridge,Puerto Nuevo,20.022410,-75.840230,Fully Operational,2,2437,2025-09-26,No,Yes
2,IC-0003,Eastern Water Treatment Plant Pue-003,Water Treatment Plant,Puerto Nuevo,20.054982,-75.793525,Fully Operational,1,2442,2025-09-29,Yes,Yes
3,IC-0004,Grid Power Substation Pue-004,Power Substation,Puerto Nuevo,19.965603,-75.893241,Fully Operational,2,3537,2025-09-20,Yes,No
4,IC-0005,Mountain Bridge Pue-005,Bridge,Puerto Nuevo,20.022899,-75.815331,Fully Operational,2,200,2025-09-19,Yes,Yes


In [70]:
map_center = (df.latitude.mean(),df.longitude.mean())
map_center

(np.float64(20.165505293333332), np.float64(-75.65761550666666))

In [71]:
df['facility_type'].unique()

<ArrowStringArray>
['Supply Distribution Center',                     'Bridge',
      'Water Treatment Plant',           'Power Substation',
           'School (Shelter)',               'Fire Station',
                    'Shelter',              'Health Clinic',
                   'Hospital',       'Communications Tower']
Length: 10, dtype: str

In [72]:
df[df['damage_severity']>4]

,facility_id,facility_name,facility_type,municipality,latitude,longitude,operational_status,damage_severity,population_served,date_last_update,has_generator,road_access
74,IC-0075,Colegio School Cos-075,School (Shelter),Costa Sur,20.099810,-75.215620,Non-Operational,5,269,2025-10-01,Yes,Limited
75,IC-0076,Mountain Bridge Cos-076,Bridge,Costa Sur,20.137094,-75.219347,Non-Operational,5,6510,2025-09-24,No,Yes
126,IC-0127,Colegio School Rin-127,School (Shelter),Rincon del Este,20.324131,-74.470232,Non-Operational,5,3151,2025-09-22,No,Limited
137,IC-0138,Tower Communications Tower Rin-138,Communications Tower,Rincon del Este,20.289238,-74.608396,Non-Operational,5,4370,2025-09-20,No,No


In [73]:
df.describe()

,latitude,longitude,damage_severity,population_served
count,150.000000,150.000000,150.000000,150.000000
mean,20.165505,-75.657616,2.393333,2664.440000
std,0.151238,0.681439,1.104577,3326.487628
min,19.957705,-76.774534,1.000000,200.000000
25%,20.036879,-75.948087,1.250000,707.000000
50%,20.106596,-75.828800,2.000000,1579.000000
75%,20.334255,-75.209873,3.000000,3139.250000
max,20.439625,-74.462293,5.000000,22400.000000


**Damage severity by municipality based on mean severity for facilities.**  
1=Least severe, 5=Most severe  
The overall highest severity by this measure is Rincon del Este, followed by Costa Sur.  

In [61]:
severity_order = df.groupby('municipality')['damage_severity'].mean().sort_values(ascending=False).reset_index()
severity_order

,municipality,damage_severity
0,Rincon del Este,3.360000
1,Costa Sur,3.200000
2,Bahia Verde,2.071429
3,Sierra Alta,2.000000
4,Puerto Nuevo,1.711111


In [62]:

access = ['Yes', 'Limited', 'No']
access_color = dict(zip(access,['green','orange','red']))

fig = px.histogram(
    df, 
    x='municipality', 
    color='road_access', 
    barmode='group', # Use 'stack' if you prefer stacked bars
    title='Road Access to Facilities by Municipality',
    labels={'municipality': 'Municipality'},
    category_orders={
        'road_access': access,
        'municipality': severity_order['municipality']
    },
    color_discrete_map=access_color
)
fig.show()

In [63]:
df['operational_status'].unique()

<ArrowStringArray>
['Fully Operational', 'Partially Operational', 'Non-Operational']
Length: 3, dtype: str

In [90]:
op_status = ['Fully Operational', 'Partially Operational', 'Non-Operational']
colors = ['green','orange','red']
op_code = ['Operational','Partly Operational','Not-operational']
stat_color = dict(zip(op_status,colors))
code = dict(zip(op_status,op_code))
stat_color

{'Fully Operational': 'green',
 'Partially Operational': 'orange',
 'Non-Operational': 'red'}

In [65]:
facility_icon = {'Supply Distribution Center':'truck', 
            'Bridge':'road',
            'Water Treatment Plant':'tint' ,
            'Power Substation':'bolt',
            'School (Shelter)':'home',
            'Fire Station':'fire-extinguisher',
            'Shelter':'home',
            'Health Clinic':'medkit',
            'Hospital':'h-square',
            'Communications Tower':'podcast'}

In [95]:
m1 = folium.Map(location=map_center, zoom_start=9, tiles="CartoDB positron")

for _, row in df.iterrows():
    folium.Marker(location=[row['latitude'],row['longitude']],
                  popup=f"{row['municipality']}\n{row['facility_name'].rsplit(maxsplit=1)[-1]}\n{row['facility_type']}\nOpStatus={code[row['operational_status']]}\nSeverity={row['damage_severity']}\nPop={row['population_served']}",
                  icon=folium.Icon(color=stat_color[row["operational_status"]],prefix='fa',icon=facility_icon[row['facility_type']])
                 ).add_to(m1)
    


   
legend_html = '''
<div style="position: fixed; 
     bottom: 50px; left: 50px; width: 200px; height: 100px; 
     border:2px solid grey; z-index:9999; font-size:14px;
     background-color:white; opacity: 0.85;">
     &nbsp; <b>Facility Status</b> <br>
     &nbsp; Operational &nbsp; <i class="fa fa-circle" style="color:green"></i><br>
     &nbsp; Partially Operational &nbsp; <i class="fa fa-circle" style="color:orange"></i><br>
     &nbsp; Not Operational &nbsp; <i class="fa fa-circle" style="color:red"></i><br>
</div>
'''
m1.get_root().html.add_child(folium.Element(legend_html))

m1  

In [74]:
m1a = folium.Map(location=map_center, zoom_start=9, tiles="CartoDB positron")

for _, row in df.iterrows():
    folium.CircleMarker(location=[row['latitude'],row['longitude']],
                  radius=3,fill=True,fill_opacity=1,color=stat_color[row["operational_status"]],
                  popup=f"{row['municipality']}\n{row['facility_name'].rsplit(maxsplit=1)[-1]}\n{row['facility_type']}\nOpStatus={code[row['operational_status']]}\nSeverity={row['damage_severity']}\nPop={row['population_served']}",
                  #icon=folium.Icon(color=cols[row['operational_status']])
                 ).add_to(m1a)
    
legend_html = '''
<div style="position: fixed; 
     bottom: 50px; left: 50px; width: 200px; height: 100px; 
     border:2px solid grey; z-index:9999; font-size:14px;
     background-color:white; opacity: 0.85;">
     &nbsp; <b>Facility Status</b> <br>
     &nbsp; Operational &nbsp; <i class="fa fa-circle" style="color:green"></i><br>
     &nbsp; Partially Operational &nbsp; <i class="fa fa-circle" style="color:orange"></i><br>
     &nbsp; Not Operational &nbsp; <i class="fa fa-circle" style="color:red"></i><br>
</div>
'''
m1a.get_root().html.add_child(folium.Element(legend_html))

m1a  

In [67]:
# Hospital and water treatment
# Show radius around hospitals and water treatment plants in a different color.

m2 = folium.Map(location=map_center, zoom_start=9, tiles="CartoDB positron")  #tiles="OpenStreetMap")

rad_hosp_miles = 15
rad_wt_miles = 30

critical = ['Hospital','Water Treatment Plant']
radius_m = [rad_hosp_miles*1609.34,rad_wt_miles*1609.34]  # Miles
col = ['blue','grey']
radius = dict(zip(critical,radius_m))
color = dict(zip(critical,col))

infr = df[df['facility_type'].isin(critical)]

for _, row in infr.iterrows():   # radius circles
    folium.Circle(
        location=([row['latitude'],row['longitude']]),
        radius=radius[row['facility_type']],
        color=None,
        fill=True,
        fill_color=color[row["facility_type"]],
        fill_opacity=0.2,
    ).add_to(m2)

for _, row in infr.iterrows():   # Dots
    folium.Circle(location=[row['latitude'],row['longitude']],
                        radius=3, fill=True, fill_opacity=1,
                        popup=f"{row['municipality']}\n{row['facility_name'].rsplit(maxsplit=1)[-1]}\n{row['facility_type']}\n{code[row['operational_status']]}\nSeverity={row['damage_severity']}\nPop={row['population_served']}\nRoad Access: {row['road_access']}",
                        color=stat_color[row["operational_status"]],
                       ).add_to(m2)

legend_html = '''
<div style="position: fixed; 
     bottom: 50px; left: 50px; width: 200px; height: 100px; 
     border:2px solid grey; z-index:9999; font-size:14px;
     background-color:white; opacity: 0.85;">
     &nbsp; <b>Vital Infrastructure</b> <br>
     &nbsp; Radius: Hospital &nbsp; <i class="fa fa-circle" style="color:grey"></i><br>
     &nbsp; Radius: Water Treatment &nbsp; <i class="fa fa-circle" style="color:blue"></i><br>
     &nbsp; Facility Status &nbsp; <i class="fa fa-circle" style="color:green"></i> <i class="fa fa-circle" style="color:orange"></i> <i class="fa fa-circle" style="color:red"></i><br>
</div>
'''
m2.get_root().html.add_child(folium.Element(legend_html))

m2

Since this is a fictional storm and location superimposed on a map of the southern tip of Cuba, it's not certain how well the radii shown in the map above reflect coverage of the landmass.

Rincon del Este's hospitals are either operational or partly so, but road access is limited to both of them. With Rin-147 operational, there should be a high priority on clearing roads in that municipality. Otherwise the closest operational hospital is Cos-086 in Costa Sur. Roads should also be cleared to the other Rincon del Este hospital, Rin-140, so that recovery can begin there and in case patients need to be transferred. Emergencies can be transferred to the western municipalities if helicopter resources are available.  

Costa Sur has two operational hospitals, with no road access to Cos-088. Cos-092 is not operational, with limited road access. Priority on restoring road access to Cos-088 and restoring operation and access to Cos-092.

Costa Sur's water treatment plants are all operationally limited, and more than 30 miles from an operational alternative. Residents should be advised to conserve water. Priorities there should include water delivery, and restoring those facilities. Rincon del Este's water treatment plant is operational, but too far to be off help to Costa Sur.  

Bahia Verde, Sierra Alta and Puerto Nuevo hospitals and water treatment plants are operational. Priorities in those municipalities should ensure road access to hospitals.  

In [68]:
m3 = folium.Map(location=map_center, zoom_start=9, tiles="CartoDB positron")
HeatMap(
    [df['latitude'],df['longitude'],df['damage_severity']],  # TODO: This might be the problem
    min_opacity=0.3,
    radius=12,
    blur=10,
    gradient={1: 'blue', 2: 'lime', 3: 'yellow', 4: 'orange', 5: 'red'}
).add_to(m3)

m3